# 08 – Query-only emergency check (Kaggle) — revision round 2, reviewer comment 4

Asks d1-3B **only the emergency question**, with the **user query alone** (no retrieved passage, no tools).
Run on every unique query of the dev set (standard + dialect; wrong-passage rows reuse the standard query)
and of the native pilot set. Temperatures/thresholds for this score are fit later on the dev `fit` split only
(`scripts/analyze_query_only.py`); nothing is tuned on native data.

Settings: Accelerator = GPU (T4), Internet = On, Input = dataset containing
`AISIS_DevSet_v1.xlsx`, `native_eval_v1.csv`, `gate_questions_v2.json`.
Output: `results_query_only_emergency.csv` → `results/query_only/results_query_only_emergency_v1.csv`.

In [ ]:
!pip install -q -U "transformers>=5.14" accelerate

In [ ]:
import glob, json, time, pandas as pd, torch
f = lambda n: glob.glob(f"/kaggle/input/**/{n}", recursive=True)[0]
dev = pd.read_excel(f("AISIS_DevSet_v1.xlsx"), sheet_name="DevSet").fillna("")
nat = pd.read_csv(f("native_eval_v1.csv")).fillna("")
Q = json.load(open(f("gate_questions_v2.json"), encoding="utf-8"))["questions"]
EMERG = {"emergency": Q["emergency"]}          # only the emergency question

rows = []
for r in dev.itertuples():                      # dev: standard + dialect (wrong_passage = same query as standard)
    rows.append(dict(source="dev", item_id=r.dev_id, version="standard", query=r.query_standard))
    rows.append(dict(source="dev", item_id=r.dev_id, version="dialect", query=r.query_dialect_synth))
for r in nat.itertuples():                      # native: one row per (base query, rendering)
    rows.append(dict(source="native", item_id=r.id, version=r.version, query=r.query))
J = pd.DataFrame(rows)
print(len(J), "queries:", J.groupby("source").size().to_dict())   # expect dev 800, native 209

In [ ]:
from transformers import AutoModel
def load(dtype): return AutoModel.from_pretrained("LiquidAI/d1-3B", trust_remote_code=True, dtype=dtype).to("cuda")
model = load(torch.float16)
t = model.system_one("test", {"q": {"type": "choice", "instructions": "pick", "criteria": {"a": "first", "b": "second"}}})
p = t["answers"]["q"]["probabilities"]; vals = list(p.values()) if isinstance(p, dict) else list(p)
if any(v != v for v in vals):
    del model; torch.cuda.empty_cache(); model = load(torch.float32); print("fell back to fp32")

In [ ]:
out, t0 = [], time.time()
for k, r in enumerate(J.to_dict("records")):
    a = model.system_one({"user_query": r["query"]}, EMERG)["answers"]["emergency"]   # query only: no passage, no tools
    out.append({**r, "pred_emergency_q": a.get("choice"), "conf_emergency_q": a.get("confidence"),
                "probs_emergency_q": json.dumps(a.get("probabilities"), ensure_ascii=False, default=str)})
    if k == 0: print(out[0])
el = time.time() - t0
print(f"{el:.1f}s, {1000*el/len(J):.0f} ms per query; peak GPU GB {torch.cuda.max_memory_allocated()/1e9:.1f}")
R = pd.DataFrame(out); R.to_csv("/kaggle/working/results_query_only_emergency.csv", index=False)
print(R.groupby(["source", "pred_emergency_q"]).size())

In [ ]:
import base64
from IPython.display import HTML
d = base64.b64encode(open("/kaggle/working/results_query_only_emergency.csv","rb").read()).decode()
HTML(f'<a download="results_query_only_emergency.csv" href="data:text/csv;base64,{d}">Download results_query_only_emergency.csv</a>')